# Popolazione per Comune per Anno

Legge i file `POSAS_<anno>_it_Comuni.csv` e costruisce una griglia:
- Righe: comuni
- Colonne: anni
- Valori: popolazione totale (riga con età=999, ultima colonna)

In [ ]:
import pandas as pd
from pathlib import Path

In [ ]:
def parse_posas_file(filepath):
    """
    Legge un file POSAS_<anno>_it_Comuni.csv (separato da spazi).
    Restituisce un DataFrame con colonne: codice, comune, eta, totale
    
    Struttura riga: CODE NOME... ETA NUM NUM ... TOTALE
    - CODE: codice ISTAT (5 cifre)
    - NOME: nome comune (può avere più parole, tutte non numeriche)
    - ETA: 0-100 oppure 999 (totale)
    - TOTALE: ultimo valore numerico della riga
    """
    records = []
    with open(filepath, encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            tokens = line.split()
            if len(tokens) < 4:
                continue
            code = tokens[0]
            # Le parole del nome sono tutti i token non numerici dopo il codice
            i = 1
            name_parts = []
            while i < len(tokens) and not tokens[i].lstrip('-').isdigit():
                name_parts.append(tokens[i])
                i += 1
            if not name_parts or i >= len(tokens):
                continue
            name = ' '.join(name_parts)
            age = int(tokens[i])
            values = tokens[i+1:]
            if not values:
                continue
            total = int(values[-1])
            records.append({'codice': code, 'comune': name, 'eta': age, 'totale': total})
    return pd.DataFrame(records)

In [ ]:
# Anni da processare (modifica l'intervallo se necessario)
ANNI = list(range(2019, 2026))  # 2019-2025
DATA_DIR = Path('/content')

risultati = {}

for anno in ANNI:
    filepath = DATA_DIR / f'POSAS_{anno}_it_Comuni.csv'
    if not filepath.exists():
        print(f'File non trovato: {filepath}')
        continue
    df = parse_posas_file(filepath)
    # Prendi solo le righe con eta=999 (totali)
    totali = df[df['eta'] == 999][['codice', 'comune', 'totale']].copy()
    totali = totali.set_index(['codice', 'comune'])['totale']
    risultati[anno] = totali
    print(f'{anno}: {len(totali)} comuni caricati')

print(f'\nAnni caricati: {list(risultati.keys())}')

In [ ]:
# Costruisci la griglia: comuni x anni
griglia = pd.DataFrame(risultati)
griglia.index.names = ['codice', 'comune']
griglia.columns.name = 'anno'

print(f'Griglia: {griglia.shape[0]} comuni x {griglia.shape[1]} anni')
griglia.head(10)

In [ ]:
# Verifica: valori mancanti (comuni presenti solo in alcuni anni)
mancanti = griglia.isnull().sum()
if mancanti.sum() > 0:
    print('Valori mancanti per anno:')
    print(mancanti[mancanti > 0])
else:
    print('Nessun valore mancante')

In [ ]:
# Salva il risultato in CSV
output_path = DATA_DIR / 'popolazione_comuni_per_anno.csv'
griglia.to_csv(output_path)
print(f'Salvato in: {output_path}')

In [ ]:
# Esempio: cerca un comune specifico
comune_cerca = 'Abano Terme'
mask = griglia.index.get_level_values('comune') == comune_cerca
if mask.any():
    print(f'Popolazione di {comune_cerca} per anno:')
    print(griglia[mask].T)
else:
    print(f'Comune "{comune_cerca}" non trovato')